# 02 — Clean & Standardize

Take the raw IRS SOI and Census ACS tables loaded by `01-ingest` and standardize them into **directed-edge staging tables** inside DuckDB.

All logic lives in SQL via `src/prepare.py`. Outputs of this stage:

| staging table | grain | purpose |
|---|---|---|
| `state_ref` | one row per location | FIPS ↔ abbrev ↔ name ↔ loc_type crosswalk |
| `irs_edges` | origin × dest × year | IRS flows, out & in measures side by side |
| `acs_edges` | origin × dest × year | ACS survey flows (migrants + MOE) |
| `nonmigrants` | state × year | IRS non-migrant (stayed-put) rows |

The final merge into `migration_flows` happens in `03-prepare`.

**Key standardization decisions** (see codebook / SOURCES.md):
- Year alignment: IRS `YYNN` → survey year = second year (IRS 2022–23 ↔ ACS 2023).
- `migration_flows` keeps **true state-to-state edges only** — summary rows (FIPS 96/97/98), overseas (59), and non-migrant self-edges are excluded.
- Non-migrants go to their **own table** for a future join to a master states table.
- All locations kept & tagged (`state`, `dc`, `territory`, `foreign`) — filterable later.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import get_connection, load_to_duckdb, quality_report, save_interim
from src.prepare import (
    build_state_ref, build_irs_edges, build_acs_edges, build_nonmigrants,
    build_county_edges, build_county_nonmigrants, build_county_income_class,
)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')
print('Raw tables present:')
print(con.execute("SELECT table_name FROM information_schema.tables WHERE table_name LIKE 'state%flow%' OR table_name='census_acs_migration' ORDER BY table_name").df().to_string(index=False))

## 1. State crosswalk
`state_ref` maps the IRS keys (FIPS / 2-letter) to the ACS keys (full name) so the two sources can be joined. It also tags each location's type. IRS summary FIPS (96/97/98) and overseas (59) are intentionally excluded — they are aggregates, not places.

In [ ]:
ref = build_state_ref(con)
print(f'{len(ref)} locations')
print(ref['loc_type'].value_counts().to_string())
ref.head()

## 2. IRS directed edges
Union all 12 outflow + 12 inflow year tables into one edge table. Each edge carries **both** the outflow-file and inflow-file measurement so we never silently pick one (the two series diverge slightly in later years). Non-migrant self-edges and summary rows are dropped here.

In [ ]:
irs_edges = build_irs_edges(con)
load_to_duckdb(irs_edges, 'irs_edges', con)
print(f'irs_edges: {len(irs_edges):,} rows, years {irs_edges.year.min()}–{irs_edges.year.max()}')
irs_edges.head()

## 3. ACS directed edges
Normalize the ACS long table: drop the two aggregate `Total` rows per origin/year, drop the origin-only `United States` aggregate, fix label casing/plurals, and resolve both endpoints to FIPS via `state_ref`.

In [ ]:
acs_edges = build_acs_edges(con)
load_to_duckdb(acs_edges, 'acs_edges', con)
print(f'acs_edges: {len(acs_edges):,} rows, years {acs_edges.year.min()}–{acs_edges.year.max()}')
acs_edges.head()

## 4. Non-migrants (stayed-put)
IRS marks non-migrants with `origin_fips == dest_fips`. Pulled into their own per-state × year table, keyed to `state_ref` for a future join to a master states table.

In [ ]:
nonmigrants = build_nonmigrants(con)
load_to_duckdb(nonmigrants, 'nonmigrants', con)
print(f'nonmigrants: {len(nonmigrants):,} rows ({len(nonmigrants)//nonmigrants.year.nunique()} states × {nonmigrants.year.nunique()} years)')
nonmigrants.head()

## 4b. County-to-county staging
Standardize the raw per-state county data (`county_raw`) into three staging tables:
- `county_edges` — directed county→county edges (out/in measures side by side), true county pairs only
- `county_nonmigrants` — one row per county with an **AGI-per-return income proxy** from its non-migrant row
- `county_income_class` — each county flagged **above/below the national median** AGI-per-return (per year)

In [ ]:
county_edges = build_county_edges(con)
load_to_duckdb(county_edges, 'county_edges', con)
print(f'county_edges: {len(county_edges):,} rows')

county_nonmigrants = build_county_nonmigrants(con)
load_to_duckdb(county_nonmigrants, 'county_nonmigrants', con)
print(f'county_nonmigrants: {len(county_nonmigrants):,} counties')

county_income_class = build_county_income_class(con)
load_to_duckdb(county_income_class, 'county_income_class', con)
med = county_income_class['national_median'].iloc[0]
print(f'county_income_class: {len(county_income_class):,} counties | national median AGI/return = ${med:,.0f}')
print(county_income_class['income_class'].value_counts().to_string())
county_income_class.head()

## 5. Quality reports

In [ ]:
_ = quality_report(irs_edges, 'irs_edges', con,
                   required_columns=['origin_fips', 'dest_fips', 'year'], max_null_pct=0.60)
_ = quality_report(acs_edges, 'acs_edges', con,
                   required_columns=['origin_fips', 'dest_fips', 'year'], max_null_pct=0.05)
_ = quality_report(nonmigrants, 'nonmigrants', con,
                   required_columns=['fips', 'year'], max_null_pct=0.01)

> Note: high null % in `irs_edges` in/out columns is expected — an edge present in the outflow file but not the inflow file (or vice versa) leaves the other side null after the full outer join. This is a real data signal, not an error.

## 6. Save interim + register provenance

In [ ]:
save_interim(ref,         cfg, 'state_ref.parquet')
save_interim(irs_edges,   cfg, 'irs_edges.parquet')
save_interim(acs_edges,   cfg, 'acs_edges.parquet')
save_interim(nonmigrants, cfg, 'nonmigrants.parquet')
save_interim(county_edges,        cfg, 'county_edges.parquet')
save_interim(county_nonmigrants,  cfg, 'county_nonmigrants.parquet')
save_interim(county_income_class, cfg, 'county_income_class.parquet')

# Register provenance using the project's existing _sources schema
# (table_name, source_name, source_url, description, retrieved_date, row_count),
# created by 01-ingest.
from datetime import date
today = date.today().isoformat()
IRS_URL = 'https://www.irs.gov/statistics/soi-tax-stats-migration-data'
ACS_URL = 'https://www.census.gov/data/tables/time-series/demo/geographic-mobility/state-to-state-migration.html'

def register(table, source_name, source_url, description, n):
    con.execute('INSERT OR REPLACE INTO _sources (table_name, source_name, source_url, description, retrieved_date, row_count) VALUES (?, ?, ?, ?, ?, ?)',
                [table, source_name, source_url, description, today, int(n)])

register('state_ref', 'Internal crosswalk', '',
         'FIPS/abbrev/name/loc_type crosswalk (50 states + DC + PR + U.S. Island Areas + Foreign). Built in src/prepare.py.', len(ref))
register('irs_edges', 'IRS SOI Migration Data', IRS_URL,
         'Directed edges, outflow+inflow measures side by side. Year=second year of IRS pair. Non-migrant self-edges and 96/97/98/59 summaries excluded.', len(irs_edges))
register('acs_edges', 'Census Bureau ACS', ACS_URL,
         'Directed edges from ACS 1-year. Two aggregate Total rows per origin/year and United States aggregate dropped. Year=survey year. No 2020 (COVID).', len(acs_edges))
register('nonmigrants', 'IRS SOI Migration Data', IRS_URL,
         'IRS rows where origin_fips==dest_fips (filed in same state both years). One row per state x year.', len(nonmigrants))

register('county_edges', 'IRS SOI Migration Data', IRS_URL,
         'Directed county-to-county edges (out/in measures side by side), true county pairs only.', len(county_edges))
register('county_nonmigrants', 'IRS SOI Migration Data', IRS_URL,
         'County non-migrant rows with AGI-per-return income proxy (agi*1000/returns).', len(county_nonmigrants))
register('county_income_class', 'IRS SOI Migration Data (derived)', IRS_URL,
         'Each county flagged above/below national median AGI-per-return, per year.', len(county_income_class))

print('\n_sources:')
print(con.execute("SELECT table_name, source_name, row_count FROM _sources WHERE table_name IN ('state_ref','irs_edges','acs_edges','nonmigrants','county_edges','county_nonmigrants','county_income_class') ORDER BY table_name").df().to_string(index=False))

---
**Next:** `03-prepare.ipynb` merges `irs_edges` + `acs_edges` into the unified `migration_flows` table and packages the export.

---
## Cleanup
Close the DuckDB connection so the write lock is released for other tools (DBCode, other notebooks).

In [ ]:
con.close()
print('connection closed')